In [2]:
# ── Cell 1: Mount Drive & Setup ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Project root on Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")
PROJECT_ROOT.mkdir(exist_ok=True)

# Data lives here (adjust if your data folder is named differently)
DATA_DIR = PROJECT_ROOT / "data"

# Create folders for checkpoints and outputs
(PROJECT_ROOT / "checkpoints").mkdir(exist_ok=True)
(PROJECT_ROOT / "notebooks").mkdir(exist_ok=True)
(PROJECT_ROOT / "submissions").mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Contents of data dir:", os.listdir(DATA_DIR))
print("Folders created: checkpoints, notebooks, submissions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/kaggle_final_competition
Data dir: /content/drive/MyDrive/kaggle_final_competition/data
Contents of data dir: ['train.csv', 'val.csv', 'test.csv', 'sample_submission.csv', 'images']
Folders created: checkpoints, notebooks, submissions


In [3]:
# ── Cell 2: EDA ──────────────────────────────────────────────────────────────
import pandas as pd
import json
from PIL import Image

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print("=== 1. num_choices distribution ===")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name}:")
    print(df["num_choices"].value_counts().sort_index())

print("\n=== 2. Subject distribution (train) ===")
print(train_df["subject"].value_counts())

print("\n=== 3. Grade distribution (train) ===")
print(train_df["grade"].value_counts().sort_index())

print("\n=== 4. Hint/Lecture availability ===")
for name, df in [("train", train_df), ("val", val_df)]:
    has_hint = df["hint"].notna().sum()
    has_lecture = df["lecture"].notna().sum()
    has_both = (df["hint"].notna() & df["lecture"].notna()).sum()
    print(f"{name}: hint={has_hint}/{len(df)}, lecture={has_lecture}/{len(df)}, both={has_both}/{len(df)}")

print("\n=== 5. Answer distribution (train) ===")
print(train_df["answer"].value_counts().sort_index())

print("\n=== 6. Test columns ===")
print(test_df.columns.tolist())

print("\n=== 7. Category/topic counts (train) ===")
print(f"Unique categories: {train_df['category'].nunique()}")
print(f"Unique topics: {train_df['topic'].nunique()}")
print(f"Unique skills: {train_df['skill'].nunique()}")
print("\nTop 10 categories:")
print(train_df["category"].value_counts().head(10))

print("\n=== 8. Sample image sizes ===")
sizes = []
for p in train_df["image_path"].head(50):
    img = Image.open(DATA_DIR / p)
    sizes.append(img.size)
sizes_df = pd.DataFrame(sizes, columns=["w","h"])
print(sizes_df.describe())

print("\n=== 9. Data dir file listing ===")
for f in sorted(os.listdir(DATA_DIR)):
    print(f)

=== 1. num_choices distribution ===

train:
num_choices
2     664
3    1552
4     783
5     110
Name: count, dtype: int64

val:
num_choices
2    244
3    508
4    252
5     44
Name: count, dtype: int64

test:
num_choices
2    272
3    438
4    260
5     38
Name: count, dtype: int64

=== 2. Subject distribution (train) ===
subject
natural science     2264
social science       768
language science      77
Name: count, dtype: int64

=== 3. Grade distribution (train) ===
grade
grade1       5
grade12     21
grade2     115
grade3     289
grade4     484
grade5     354
grade6     611
grade7     545
grade8     685
Name: count, dtype: int64

=== 4. Hint/Lecture availability ===
train: hint=2385/3109, lecture=2669/3109, both=2128/3109
val: hint=816/1048, lecture=915/1048, both=722/1048

=== 5. Answer distribution (train) ===
answer
0    1124
1    1028
2     737
3     204
4      16
Name: count, dtype: int64

=== 6. Test columns ===
['id', 'image_path', 'question', 'choices', 'num_choices', 'hint',

In [4]:
print("Train rows:", len(train_df))
print("Test IDs sample:", test_df["id"].head())
print("Sample submission shape:", pd.read_csv(DATA_DIR / "sample_submission.csv").shape)

Train rows: 3109
Test IDs sample: 0    test_01750
1    test_00128
2    test_02891
3    test_02425
4    test_00930
Name: id, dtype: object
Sample submission shape: (1008, 2)
